<a href="https://colab.research.google.com/github/rohkdev/ClimateModelingResearch/blob/main/Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

print("Connecting to NOAA Global Historical Climatology Network...")

# 1. Define the widths and column names for ghcnd-stations.txt (fixed-width file)
STATION_WIDTHS = [11, 9, 10, 7, 3, 31]
STATION_COLS = ['station_id', 'latitude', 'longitude', 'elevation', 'state', 'name']

# 2. Define the widths and column names for ghcnd-inventory.txt (fixed-width file)
INVENTORY_WIDTHS = [11, 9, 10, 5, 5, 5]
INVENTORY_COLS = ['station_id', 'latitude', 'longitude', 'element', 'first_year', 'last_year']

print("Loading stations metadata...")
df_stations = pd.read_fwf("https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt",
                          widths=STATION_WIDTHS, names=STATION_COLS)

print("Loading inventory metadata...")
df_inventory = pd.read_fwf("https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-inventory.txt",
                           widths=INVENTORY_WIDTHS, names=INVENTORY_COLS)

# 3. Filter stations to California (CA) only
print("Filtering for California stations...")
df_ca_stations = df_stations[df_stations['state'] == 'CA'].copy()

# 4. Filter inventory for California stations that have TMAX or TMIN elements
df_ca_inventory = df_inventory[
    (df_inventory['station_id'].isin(df_ca_stations['station_id'])) &
    (df_inventory['element'].isin(['TMAX', 'TMIN']))
]

# 5. Pivot the inventory to get first_year and last_year for TMAX and TMIN on the same row
df_pivot = df_ca_inventory.pivot(index='station_id', columns='element', values=['first_year', 'last_year'])
df_pivot.columns = [f"available_{elem}_{stat}" for stat, elem in df_pivot.columns]
df_pivot = df_pivot.reset_index()

# 6. Merge the station info (name, lat, lon, elevation) with the data timeline info
candidate_station_table = pd.merge(df_ca_stations, df_pivot, on='station_id', how='inner')

# 7. Drop stations that are missing either TMAX or TMIN completely
candidate_station_table = candidate_station_table[
    candidate_station_table['available_TMAX_first_year'].notna() &
    candidate_station_table['available_TMIN_first_year'].notna()
].copy()

# 8. Save the clean dataset to a CSV file
output_file = "candidate_station_table.csv"
candidate_station_table.to_csv(output_file, index=False)

print(f"\nSuccess! Found {len(candidate_station_table)} California stations with both TMAX and TMIN records.")
print(f"Dataset saved cleanly as: '{output_file}'")

Connecting to NOAA Global Historical Climatology Network...
Loading stations metadata...
Loading inventory metadata...
Filtering for California stations...

Success! Found 1242 California stations with both TMAX and TMIN records.
Dataset saved cleanly as: 'candidate_station_table.csv'


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# User settings
# ============================================================

DATA_DIR = Path(".")  # change if needed

DAILY_FILE = DATA_DIR / "Temperature Data.csv"

# If you already have Setup.ipynb's output, use this.
# Otherwise, set USE_EXISTING_SETUP_TABLE = False and use the NOAA URLs below.
SETUP_CANDIDATE_FILE = DATA_DIR / "candidate_station_table.csv"
USE_EXISTING_SETUP_TABLE = SETUP_CANDIDATE_FILE.exists()

OUTPUT_FILE = DATA_DIR / "Research_Dataset-candidate_station_table.csv"

STUDY_START_YEAR = 1970
STUDY_END_YEAR = None  # None means use latest year in daily file

SUMMER_MONTHS = [6, 7, 8, 9]
EXPECTED_SUMMER_DAYS = 122

# A "usable year" means both TMAX and TMIN have at least this many valid summer days.
MIN_VALID_SUMMER_FRACTION = 0.90
MIN_VALID_SUMMER_DAYS = int(np.ceil(EXPECTED_SUMMER_DAYS * MIN_VALID_SUMMER_FRACTION))

print("Minimum valid summer days required per variable:", MIN_VALID_SUMMER_DAYS)


# ============================================================
# 1. Load or reconstruct the basic candidate station table
# ============================================================

if USE_EXISTING_SETUP_TABLE:
    print(f"Loading existing setup table: {SETUP_CANDIDATE_FILE}")
    candidate_station_table = pd.read_csv(SETUP_CANDIDATE_FILE)

else:
    print("Reconstructing setup table from NOAA metadata...")

    STATION_WIDTHS = [11, 9, 10, 7, 3, 31]
    STATION_COLS = ["station_id", "latitude", "longitude", "elevation", "state", "name"]

    INVENTORY_WIDTHS = [11, 9, 10, 5, 5, 5]
    INVENTORY_COLS = ["station_id", "latitude", "longitude", "element", "first_year", "last_year"]

    stations_url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
    inventory_url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-inventory.txt"

    df_stations = pd.read_fwf(
        stations_url,
        widths=STATION_WIDTHS,
        names=STATION_COLS
    )

    df_inventory = pd.read_fwf(
        inventory_url,
        widths=INVENTORY_WIDTHS,
        names=INVENTORY_COLS
    )

    # California stations only
    df_ca_stations = df_stations[df_stations["state"] == "CA"].copy()

    # Inventory rows for TMAX/TMIN only
    df_ca_inventory = df_inventory[
        (df_inventory["station_id"].isin(df_ca_stations["station_id"])) &
        (df_inventory["element"].isin(["TMAX", "TMIN"]))
    ].copy()

    # Pivot inventory so TMAX/TMIN year ranges are on the same row
    df_pivot = df_ca_inventory.pivot_table(
        index="station_id",
        columns="element",
        values=["first_year", "last_year"],
        aggfunc="first"
    )

    df_pivot.columns = [
        f"available_{elem}_{stat}"
        for stat, elem in df_pivot.columns
    ]

    df_pivot = df_pivot.reset_index()

    candidate_station_table = pd.merge(
        df_ca_stations,
        df_pivot,
        on="station_id",
        how="inner"
    )

    # Keep stations that have both TMAX and TMIN in the inventory
    candidate_station_table = candidate_station_table[
        candidate_station_table["available_TMAX_first_year"].notna() &
        candidate_station_table["available_TMIN_first_year"].notna()
    ].copy()


# Make sure expected names exist
candidate_station_table["station_id"] = candidate_station_table["station_id"].astype(str)

print("Basic candidate table shape:", candidate_station_table.shape)
print(candidate_station_table.head())




Minimum valid summer days required per variable: 110
Loading existing setup table: candidate_station_table.csv
Basic candidate table shape: (1242, 10)
    station_id  latitude  longitude  elevation state         name  \
0  USC00040024   34.5833  -117.4167      869.0    CA     ADELANTO   
1  USC00040029   41.1933  -120.9444     1280.8    CA      ADIN RS   
2  USC00040064   37.8667  -122.0333      125.0    CA    ALAMO 1 N   
3  USC00040080   34.3500  -118.0500     1310.6    CA  ALDER CREEK   
4  USC00040088   40.1833  -123.6167      139.9    CA   ALDERPOINT   

   available_TMAX_first_year  available_TMIN_first_year  \
0                     1967.0                     1967.0   
1                     1894.0                     1894.0   
2                     1968.0                     1968.0   
3                     1925.0                     1925.0   
4                     1941.0                     1941.0   

   available_TMAX_last_year  available_TMIN_last_year  
0                    19

In [ ]:
# ============================================================
# 2. Process daily file in chunks to compute summer coverage
# ============================================================

coverage_chunks = []

usecols = [
    "station_id",
    "year",
    "month",
    "day",
    "TMAX",
    "TMIN",
]

# Some versions of the file may not have year/month, so we handle that below.
for i, chunk in enumerate(pd.read_csv(DAILY_FILE, chunksize=500_000, low_memory=False), start=1):

    # Keep only columns that exist
    existing_cols = [c for c in usecols if c in chunk.columns]
    chunk = chunk[existing_cols].copy()

    # Ensure station_id is string
    chunk["station_id"] = chunk["station_id"].astype(str)

    # Create date/year/month if needed
    if "day" in chunk.columns:
        chunk["day"] = pd.to_datetime(chunk["day"], errors="coerce")

    if "year" not in chunk.columns:
        chunk["year"] = chunk["day"].dt.year

    if "month" not in chunk.columns:
        chunk["month"] = chunk["day"].dt.month

    # Study-period filter
    chunk = chunk[chunk["year"] >= STUDY_START_YEAR]

    if STUDY_END_YEAR is not None:
        chunk = chunk[chunk["year"] <= STUDY_END_YEAR]

    # Summer only
    chunk = chunk[chunk["month"].isin(SUMMER_MONTHS)]

    if chunk.empty:
        continue

    # Convert to numeric just in case
    chunk["TMAX"] = pd.to_numeric(chunk["TMAX"], errors="coerce")
    chunk["TMIN"] = pd.to_numeric(chunk["TMIN"], errors="coerce")

    # Aggregate to station-year coverage
    cov = (
        chunk
        .groupby(["station_id", "year"])
        .agg(
            valid_summer_TMAX_days=("TMAX", lambda x: x.notna().sum()),
            valid_summer_TMIN_days=("TMIN", lambda x: x.notna().sum()),
        )
        .reset_index()
    )

    coverage_chunks.append(cov)

    if i % 10 == 0:
        print(f"Processed {i} chunks...")

coverage = pd.concat(coverage_chunks, ignore_index=True)

# If a station-year got split across chunks, combine counts
coverage = (
    coverage
    .groupby(["station_id", "year"], as_index=False)
    .agg(
        valid_summer_TMAX_days=("valid_summer_TMAX_days", "sum"),
        valid_summer_TMIN_days=("valid_summer_TMIN_days", "sum"),
    )
)

coverage["percent_missing_summer_TMAX_year"] = (
    100 * (1 - coverage["valid_summer_TMAX_days"] / EXPECTED_SUMMER_DAYS)
)

coverage["percent_missing_summer_TMIN_year"] = (
    100 * (1 - coverage["valid_summer_TMIN_days"] / EXPECTED_SUMMER_DAYS)
)

coverage["usable_year"] = (
    (coverage["valid_summer_TMAX_days"] >= MIN_VALID_SUMMER_DAYS) &
    (coverage["valid_summer_TMIN_days"] >= MIN_VALID_SUMMER_DAYS)
)

print("Coverage table shape:", coverage.shape)
print(coverage.head())


# ============================================================
# 3. Summarize coverage to station level
# ============================================================

station_coverage = (
    coverage
    .groupby("station_id")
    .agg(
        first_year_daily=("year", "min"),
        last_year_daily=("year", "max"),
        number_of_years_with_any_summer_data=("year", "nunique"),
        number_of_usable_years=("usable_year", "sum"),
        mean_valid_summer_TMAX_days=("valid_summer_TMAX_days", "mean"),
        mean_valid_summer_TMIN_days=("valid_summer_TMIN_days", "mean"),
        percent_missing_summer_TMAX=("percent_missing_summer_TMAX_year", "mean"),
        percent_missing_summer_TMIN=("percent_missing_summer_TMIN_year", "mean"),
    )
    .reset_index()
)

# Optional: also compute missingness only over usable/full-ish years.
# For Task_1/Task_2, the columns above are the ones being used.

print("Station coverage shape:", station_coverage.shape)
print(station_coverage.head())


# ============================================================
# 4. Merge station metadata + coverage diagnostics
# ============================================================

research_candidate_table = candidate_station_table.merge(
    station_coverage,
    on="station_id",
    how="left"
)

# Fill stations with no daily data in the processed daily file
research_candidate_table["number_of_usable_years"] = (
    research_candidate_table["number_of_usable_years"]
    .fillna(0)
    .astype(int)
)

# Make missingness 100% for stations with no daily records in the file
for col in ["percent_missing_summer_TMAX", "percent_missing_summer_TMIN"]:
    research_candidate_table[col] = research_candidate_table[col].fillna(100.0)

# Useful boolean matching Rohit's filtering logic
research_candidate_table["passes_task1_filters"] = (
    (research_candidate_table["number_of_usable_years"] >= 30) &
    (research_candidate_table["percent_missing_summer_TMAX"] <= 10) &
    (research_candidate_table["percent_missing_summer_TMIN"] <= 10)
)

research_candidate_table["passes_task2_filters"] = (
    research_candidate_table["passes_task1_filters"] &
    (research_candidate_table["elevation"] != -999.9)
)

# Save with the exact filename his notebooks expect
research_candidate_table.to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)
print("Total candidate stations:", len(research_candidate_table))
print("Pass Task 1 filters:", research_candidate_table["passes_task1_filters"].sum())
print("Pass Task 2 filters:", research_candidate_table["passes_task2_filters"].sum())

print("\nColumns:")
print(research_candidate_table.columns.tolist())

print("\nPreview:")
print(research_candidate_table.head())

Processed 10 chunks...
Processed 20 chunks...
Processed 30 chunks...
Coverage table shape: (29186, 7)
    station_id  year  valid_summer_TMAX_days  valid_summer_TMIN_days  \
0  USC00040024  1975                      30                      30   
1  USC00040029  1970                     122                     122   
2  USC00040029  1971                     121                     122   
3  USC00040029  1972                     119                     121   
4  USC00040029  1973                     121                     121   

   percent_missing_summer_TMAX_year  percent_missing_summer_TMIN_year  \
0                         75.409836                         75.409836   
1                          0.000000                          0.000000   
2                          0.819672                          0.000000   
3                          2.459016                          0.819672   
4                          0.819672                          0.819672   

   usable_year  
0        